# ADMET · Excretion assessment for rapamycin and its rapalogs

Sixth in the series (potency → **A** → **D** → **M** → **E**xcretion). Same molecules throughout.
Claims are grounded in primary literature (PubMed), cited **ACS style** in Step 6.

## Purpose

**Excretion** asks: *how does the drug leave the body* — by which route (renal vs. biliary/fecal),
how fast (clearance), and with what half-life? For the rapalogs, elimination is **biliary/fecal after
CYP3A4 metabolism**, with **very long half-lives** and low clearance.

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import io, zipfile, requests
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, rdFingerprintGenerator
from chembl_webresource_client.new_client import new_client
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, r2_score

# Notebook is stripped before push; each step's result is saved as a PNG here.
RESULTS = Path("../result_rapamycin_ADMET_Excretion"); RESULTS.mkdir(exist_ok=True)
def save_fig(fig, name): fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight")
def save_img(obj, name):
    p = RESULTS / name
    obj.save(p) if hasattr(obj, "save") else p.write_bytes(obj.data)
def save_df_png(df, name, title=None):
    fig, ax = plt.subplots(figsize=(min(2 + len(df.columns) * 1.7, 18), 0.7 + 0.45 * len(df))); ax.axis("off")
    if title: ax.set_title(title, fontsize=10, loc="left")
    t = ax.table(cellText=df.astype(str).values, colLabels=list(df.columns), loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1, 1.4); t.auto_set_column_width(range(len(df.columns)))
    fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight"); plt.close(fig)

mol_client = new_client.molecule
RAPALOGS = {"Sirolimus (rapamycin)": "CHEMBL413", "Everolimus": "CHEMBL1908360",
            "Temsirolimus": "CHEMBL1201182", "Ridaforolimus": "CHEMBL2103839",
            "Zotarolimus": "CHEMBL219410"}
smiles = {n: mol_client.filter(molecule_chembl_id=c).only(["molecule_structures"])[0]
          ["molecule_structures"]["canonical_smiles"] for n, c in RAPALOGS.items()}
mols = {n: Chem.MolFromSmiles(s) for n, s in smiles.items()}

# Therapeutics Data Commons 'admet_group' benchmark (Huang et al. 2022), cached (git-ignored)
ZIP = Path("../data/tdc_admet_group.zip")
if not ZIP.exists():
    ZIP.write_bytes(requests.get("https://dataverse.harvard.edu/api/access/datafile/4426004",
                                 headers={"User-Agent": "Mozilla/5.0"}, timeout=120).content)
_zf = zipfile.ZipFile(ZIP)
_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def _bv(s):
    m = Chem.MolFromSmiles(s); return _GEN.GetFingerprint(m) if m else None
def _arr(bv):
    a = np.zeros((2048,), np.uint8); DataStructs.ConvertToNumpyArray(bv, a); return a
def _load(name):
    d = pd.concat([pd.read_csv(_zf.open(f"admet_group/{name}/{f}")) for f in ("train_val.csv", "test.csv")], ignore_index=True)
    return d[["Drug", "Y"]].dropna()
rap_bv = {n: _GEN.GetFingerprint(m) for n, m in mols.items()}

def run_models(models, pred_png, ad_png):
    "models: list of (label, tdc_dataset, task in clf/reg). Trains RF, predicts rapalogs + applicability domain."
    rows = {n.split(" ")[0]: {"rapalog": n.split(" ")[0]} for n in rap_bv}; cols = []
    for label, ds, task in models:
        d = _load(ds); bvs = [_bv(s) for s in d["Drug"]]
        keep = [i for i, b in enumerate(bvs) if b is not None]; bvs = [bvs[i] for i in keep]
        X = np.array([_arr(b) for b in bvs]); y = d["Y"].to_numpy()[keep]
        if task == "clf":
            m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
            proba = cross_val_predict(m, X, y, cv=5, method="predict_proba", n_jobs=-1)[:, 1]
            metric = f"ROC-AUC={roc_auc_score(y, proba):.2f}"; m.fit(X, y); pr = lambda a: round(m.predict_proba(a)[0, 1], 2)
        else:
            m = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            p = cross_val_predict(m, X, y, cv=5, n_jobs=-1); metric = f"R2={r2_score(y, p):.2f}"; m.fit(X, y); pr = lambda a: round(float(m.predict(a)[0]), 2)
        print(f"{label:26s} ({ds:28s}) n={len(y):5d}  CV {metric}")
        cols.append(label)
        for n, bv in rap_bv.items():
            k = n.split(" ")[0]
            rows[k][label] = pr(_arr(bv).reshape(1, -1))
            rows[k][label + " AD"] = round(max(DataStructs.BulkTanimotoSimilarity(bv, bvs)), 2)
    df = pd.DataFrame(rows.values())
    save_df_png(df, pred_png, "de-novo ML predictions + applicability domain (max Tanimoto to training set)")
    fig, ax = plt.subplots(figsize=(9.5, 4)); x = np.arange(len(df)); w = 0.8 / len(cols)
    for i, label in enumerate(cols):
        ax.bar(x + (i - (len(cols) - 1) / 2) * w, df[label + " AD"], w, label=label.split(" ")[0][:16])
    ax.axhline(0.3, color="red", ls="--", label="in-domain ~0.3")
    ax.set_xticks(x); ax.set_xticklabels(df["rapalog"], rotation=20)
    ax.set(ylabel="max Tanimoto to training set", ylim=(0, 1.05),
           title="Applicability domain of de-novo ML models for the rapalogs")
    ax.legend(fontsize=7, ncol=2); plt.tight_layout(); save_fig(fig, ad_png); plt.show()
    return df

print("Setup ready.")

## Step 1 — Honest caveat first

**Half-life and clearance are the hardest ADMET endpoints to predict from structure** — they are
*emergent, whole-body* parameters, not molecular properties. As we will see below, a fingerprint model
of terminal half-life is **barely learnable at all** (near-zero, even negative, cross-validated R²), and
such models tend to **under-predict** unusually long half-lives — we test exactly that against the
rapalogs' ~62 h. The **900 Da problem** compounds this: clearance QSARs are trained on small drug-like
molecules. So structure-only excretion prediction is unreliable; **measured pharmacokinetics is
decisive**, and in-silico mainly *demonstrates* the shortfall.

In [ ]:
tbl = pd.DataFrame({"MW (Da)": {n: round(Descriptors.MolWt(m)) for n, m in mols.items()},
                    "cLogP": {n: round(Descriptors.MolLogP(m), 1) for n, m in mols.items()}})
print(tbl.to_string())
save_df_png(tbl.reset_index().rename(columns={"index": "drug"}), "step1_properties.png", "Step 1 - the molecules")
grid = Draw.MolsToGridImage(list(mols.values()), legends=list(mols.keys()), molsPerRow=3, subImgSize=(300, 230))
save_img(grid, "step1_structures.png"); grid

## Step 2 — A rigorous in-silico excretion stack (methods; not run here)

| Endpoint | In-silico model | Trustworthy for rapalogs? | Reference / data |
|---|---|---|---|
| **Terminal half-life** (t½) | QSAR / ML regression | ❌ out of domain; long t½ under-predicted | Obach et al., 2008 (data); Gombar & Hall, 2013 |
| **Systemic / hepatic clearance** | QSAR / ML regression | ❌ out of domain; metabolism-limited low CL | Gombar & Hall, 2013; Obach et al., 2008 |
| **Route (renal vs. biliary)** | (mechanistic / measured) | — biliary/fecal is measured | MacDonald et al., 2000 |

> **Dominant excretory mode: biliary → fecal elimination of CYP3A4 metabolites**, with minimal renal
> excretion and a long terminal half-life. Half-life/clearance are *emergent* whole-body parameters —
> the hardest to predict from structure alone.

**What we run in Step 4:** **half-life** and **hepatocyte/microsome clearance** regressors (TDC data),
predicted for the rapalogs with applicability domain, and compared to the measured sirolimus t½ ≈ 62 h.

In [ ]:
measured = pd.DataFrame([
    {"drug": "Sirolimus", "route": "biliary/fecal (~91% feces, ~2% urine)", "terminal_t_half": "~62 h",
     "clearance": "low (metabolism-limited, CYP3A4)", "ref": "MacDonald 2000; Mahalati 2001"},
    {"drug": "Everolimus", "route": "biliary/fecal", "terminal_t_half": "~30 h",
     "clearance": "low (CYP3A4)", "ref": "Kirchner 2004"},
    {"drug": "Temsirolimus", "route": "biliary/fecal", "terminal_t_half": "~17 h (sirolimus ~55 h)",
     "clearance": "low (CYP3A4)", "ref": "class / label"},
    {"drug": "Ridaforolimus", "route": "biliary/fecal", "terminal_t_half": "long", "clearance": "low",
     "ref": "class / label"},
    {"drug": "Zotarolimus", "route": "biliary/fecal (local stent use)", "terminal_t_half": "long",
     "clearance": "low", "ref": "class / label"},
])
save_df_png(measured, "step3_measured_excretion.png", "Step 3 - measured excretion")
measured

## Step 3 — Measured excretion (above) and Step 4 — run the ML stack

**Measured summary:** the rapalogs are eliminated **biliary/fecally after CYP3A4 metabolism** (renal
excretion is minor), with **low clearance** and **long half-lives** (sirolimus t½ ≈ 62 h). Now we test
whether structure-only ML reproduces these.

In [ ]:
ml = run_models([("Half-life (h)", "half_life_obach", "reg"),
                 ("Hepatocyte CL (log)", "clearance_hepatocyte_az", "reg"),
                 ("Microsome CL (log)", "clearance_microsome_az", "reg")],
                "step4_ml_excretion_predictions.png", "step4_applicability_domain.png")
ml

### What the run shows, and caveats (Step 5)

- **Half-life** is essentially **not learnable from fingerprints** here — the cross-validated R² is near
  zero (even negative), so the model is not usable *regardless* of domain. Note the rapalogs actually
  fall *inside* the training neighbourhood (several sit in the Obach human-PK database, Tanimoto ~0.8–1.0),
  yet the predicted t½ (~26–41 h) still **under-predicts** the measured sirolimus t½ ≈ 62 h. Long
  half-lives are intrinsically hard to predict.
- **Clearance** (hepatocyte / microsome) models are weak (low R²); the rapalogs' true clearance is
  **metabolism-limited (CYP3A4)** and low — an emergent property structure alone can't fix.

**Caveats.** (i) Half-life and clearance are *whole-body, emergent* PK parameters — the least
structure-predictable ADMET endpoints. (ii) Excretion **route** (biliary vs. renal) is not a fingerprint
QSAR output; it is measured. (iii) **Measured data is decisive**; here in-silico mainly *demonstrates* its
own shortfall.

## Step 6 — Citations (ACS style)

Bibliographic metadata retrieved from **PubMed**.

1. MacDonald, A.; Scarola, J.; Burke, J. T.; Zimmerman, J. J. Clinical Pharmacokinetics and
   Therapeutic Drug Monitoring of Sirolimus. *Clin. Ther.* **2000**, *22* (Suppl. B), B101–B121.
   https://doi.org/10.1016/S0149-2918(00)89027-X.
2. Mahalati, K.; Kahan, B. D. Clinical Pharmacokinetics of Sirolimus. *Clin. Pharmacokinet.*
   **2001**, *40* (8), 573–585. https://doi.org/10.2165/00003088-200140080-00002.
3. Kirchner, G. I.; Meier-Wiedenbach, I.; Manns, M. P. Clinical Pharmacokinetics of Everolimus.
   *Clin. Pharmacokinet.* **2004**, *43* (2), 83–95. https://doi.org/10.2165/00003088-200443020-00002.
4. Obach, R. S.; Lombardo, F.; Waters, N. J. Trend Analysis of a Database of Intravenous
   Pharmacokinetic Parameters in Humans for 670 Drug Compounds. *Drug Metab. Dispos.* **2008**,
   *36* (7), 1385–1405. https://doi.org/10.1124/dmd.108.020479.
5. Gombar, V. K.; Hall, S. D. Quantitative Structure–Activity Relationship Models of Clinical
   Pharmacokinetics: Clearance and Volume of Distribution. *J. Chem. Inf. Model.* **2013**, *53* (4),
   948–957. https://doi.org/10.1021/ci400001u.
6. Huang, K.; Fu, T.; Gao, W.; Zhao, Y.; Roohani, Y.; Leskovec, J.; Coley, C. W.; Xiao, C.; Sun, J.;
   Zitnik, M. Artificial Intelligence Foundation for Therapeutic Science. *Nat. Chem. Biol.* **2022**,
   *18* (10), 1033–1036. https://doi.org/10.1038/s41589-022-01131-2.

*Attribution: PubMed. Training data are the TDC `admet_group` benchmark (half_life_obach,
clearance_hepatocyte_az, clearance_microsome_az).*